# PNAD Income Analysis

This notebook runs the packaged PNAD income pipeline and exports the reproducible analysis products under `outputs/`.

Install the project once from the repository root with:

```bash
python -m pip install -e ".[notebooks]"
```


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from pnad_income.analysis import annual_inequality_indices, combine_gini_references
from pnad_income.outputs import export_analysis_outputs
from pnad_income.pipeline import PipelineConfig, pipeline_overview, run_pipeline
from pnad_income.plotting import plot_information_indices, plot_primary_indices, plot_zanardi

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "dados_refined"
OUTPUT_DIR = ROOT / "outputs"
REFERENCE_DIR = ROOT / "metadata" / "gini_references"

CONFIG = PipelineConfig(
    database_path=DATA_DIR,
    start_year=1976,
    end_year=2025,
    apply_manual_outlier_cuts=False,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


## Pipeline


In [ ]:
results = run_pipeline(CONFIG)

display(pipeline_overview(results))
display(results.summary)


## Inequality measures


In [ ]:
indices = annual_inequality_indices(results.panel, value_col="income", atkinson_epsilon=0.5)
display(indices)

for figure in (
    plot_primary_indices(indices),
    plot_zanardi(indices),
    plot_information_indices(indices),
):
    display(figure)
    plt.close(figure)


## External Gini validation

Any documented CSV files placed in `metadata/gini_references/` are included automatically.


In [ ]:
reference_files = sorted(REFERENCE_DIR.glob("*.csv"))
gini_references = (
    combine_gini_references(reference_files)
    if reference_files
    else pd.DataFrame(columns=["year", "gini", "source"])
)

if gini_references.empty:
    print("No external Gini reference CSV files found.")
else:
    display(gini_references)


## Export


In [ ]:
manifest = export_analysis_outputs(
    results,
    OUTPUT_DIR,
    gini_references=gini_references,
)

display(manifest)
